# Datos diarios: validación fuera de muestra con 15+ años

**Autor:** Samuel  
**Institución:** Tecnológico de Monterrey  
**Área:** Pairs trading en Forex  
**Notebooks previos:** Backtest-Corregido, Beta-Anclado-OU, Remuestreo-Costos, Amplitud, Eventos

Los resultados horarios dependen de ~2 años de datos: al refrescarlos, el bruto cambió ~40%. Aquí repetimos el procedimiento de *Amplitud* con datos **diarios desde 2010**, y separamos la muestra en dos bloques: las decisiones de diseño se toman con el primero y el segundo se evalúa **una sola vez**, sin ajustar nada después.

## Estructura de este Notebook
1. Fase 0: Datos diarios y corte en dos bloques
2. Fase 1: Motor (adaptado a barras diarias) y parámetros
3. Fase 2: Selección de canastas y operación en el bloque de diseño
4. Fase 3: Placebo del proceso completo
5. Fase 4: Evaluación única en el bloque de prueba
6. Conclusión

## Fase 0: Datos diarios y corte en 2 bloques
Descarga de 18 divisas diarias desde 2010, filtro de cobertura, verificación de saltos y separación en bloque de diseño (≈ 2010–2019) y bloque de prueba (≈ 2020–2026). *Entrega:* `data`, `data_sel`, `data_test`.

In [2]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings
from itertools import combinations
from scipy import stats
warnings.filterwarnings('ignore')
import yfinance as yf
try:
    from numba import njit
except ImportError:
    njit = lambda f: f

In [3]:
tickers = {
    'Baht_TH': 'USDTHB=X',  'Euro': 'EURUSD=X',     'Shekel_IL': 'USDILS=X', 'Won_KR': 'USDKRW=X',
    'Dolar_NZ': 'NZDUSD=X', 'Dolar_AUS': 'AUDUSD=X', 'Corona_CZ': 'USDCZK=X', 'Corona_SE': 'USDSEK=X',
    'Franco_SUI': 'USDCHF=X', 'Forinto_HU': 'USDHUF=X', 'Dolar_SG': 'USDSGD=X', 'Yen': 'USDJPY=X',
    'Dolar_CAN': 'USDCAD=X', 'Libra': 'GBPUSD=X',    'Peso_MX': 'USDMXN=X',  'Corona_NO': 'USDNOK=X',
    'Zloty_PL': 'USDPLN=X', 'Rand_ZA': 'USDZAR=X',
}
raw_data = yf.download(list(tickers.values()), start='2010-01-01', interval='1d',
                       progress=False, auto_adjust=True)['Close']
print(raw_data.shape)

(4387, 18)


In [4]:
t2n = {v: k for k, v in tickers.items()}
raw = raw_data.rename(columns=t2n)[list(tickers)]
if raw.index.tz is not None:
    raw.index = raw.index.tz_localize(None)
raw.index = raw.index.normalize()
raw = raw[~raw.index.duplicated()].sort_index()

cob = pd.DataFrame({'primer_dato': raw.apply(lambda s: s.first_valid_index()),
                    'faltantes_%': (raw.isna().mean() * 100).round(2)})
display(cob.sort_values('primer_dato'))

INICIO_MAX = pd.Timestamp('2012-01-01')           # exigimos historia desde, a más tardar, 2012
aceptadas = [c for c in raw.columns if cob.loc[c, 'primer_dato'] <= INICIO_MAX and cob.loc[c, 'faltantes_%'] < 3]
print("Descartadas:", [c for c in raw.columns if c not in aceptadas])

data = raw[aceptadas].ffill(limit=3).dropna()     # ffill solo hasta 3 días (festivos); el resto se descarta
print(data.shape, data.index.min().date(), '->', data.index.max().date())

,primer_dato,faltantes_%
Ticker,,
Baht_TH,2010-01-01,0.09
Corona_NO,2010-01-01,0.09
Peso_MX,2010-01-01,0.00
Libra,2010-01-01,0.09
Dolar_CAN,2010-01-01,0.11
Yen,2010-01-01,0.09
Dolar_SG,2010-01-01,0.09
Forinto_HU,2010-01-01,0.14
Franco_SUI,2010-01-01,0.16


Descartadas: []
(4387, 18) 2010-01-01 -> 2026-11-03


In [5]:
saltos = np.log(data).diff().abs().max().sort_values(ascending=False)
print("Mayor movimiento diario por divisa (log):")
print(saltos.round(3).head(6))

Mayor movimiento diario por divisa (log):
Ticker
Corona_NO     0.419
Rand_ZA       0.204
Franco_SUI    0.176
Shekel_IL     0.104
Peso_MX       0.083
Libra         0.079
dtype: float64


In [6]:
FECHA_CORTE = pd.Timestamp('2019-12-31')
data_sel  = data[data.index <= FECHA_CORTE]      # bloque de diseño: aquí se toman TODAS las decisiones
data_test = data[data.index >  FECHA_CORTE]      # bloque de prueba: se evalúa UNA sola vez, al final
print(f"Diseño: {len(data_sel)} días ({data_sel.index.min().date()} → {data_sel.index.max().date()})")
print(f"Prueba: {len(data_test)} días ({data_test.index.min().date()} → {data_test.index.max().date()})")
data.to_pickle('data_diaria.pkl')

Diseño: 2607 días (2010-01-01 → 2019-12-31)
Prueba: 1780 días (2020-01-01 → 2026-11-03)


## Fase 1: Motor y parámetros (adaptados a barras diarias)

Reutilizamos el motor de *Amplitud*, pero ahora cada barra es **un día** y no una hora. Esto cambia el significado de tres cosas:
- Las ventanas siguen siendo de 120 días de entrenamiento y 30 de prueba, pero ahora hay ~79 ventanas en el bloque de diseño en lugar de 16.
- El retraso de una barra es **un día completo** (muy conservador: equivale a ejecutar al cierre del día siguiente).
- La vida media del spread se mide en **días**, y su límite máximo (`HL_MAX`) sale de las horas (360) a un valor provisional de 30 días, que se revisa en la Fase 2 mirando **solo** el bloque de diseño.

El costo se mantiene en 1 pb por pata y por lado, para poder comparar con los resultados horarios. En la Fase 4 se prueba la sensibilidad a costos mayores.

*Entrega:* parámetros, funciones del motor y la selección de canastas calculada sobre `data_sel`.

In [8]:
TEST_DAYS, TRAIN_DAYS, MIN_TRAIN = 30, 120, 240   # barras diarias: 120 de entrenamiento, 30 de prueba
STOP = 3.5                                        # stop en sigmas
HL_MAX = 30                                       # vida media máxima aceptada, en DÍAS (provisional: se revisa en la Fase 2)
COST_BPS = 1.0                                    # por pata y por lado
N_MAX, JACC_MAX = 20, 0.4                         # canastas por ventana; traslape máximo entre canastas

In [9]:
TEST_DAYS, TRAIN_DAYS, MIN_TRAIN = 30, 120, 240   # barras diarias: 120 de entrenamiento, 30 de prueba
STOP = 3.5                                        # stop en sigmas
HL_MAX = 30                                       # vida media máxima aceptada, en DÍAS (provisional: se revisa en la Fase 2)
COST_BPS = 1.0                                    # por pata y por lado
N_MAX, JACC_MAX = 20, 0.4                         # canastas por ventana; traslape máximo entre canastas

In [14]:
def fit_ou(s):
    x, y = s[:-1], s[1:]
    b, a = np.polyfit(x, y, 1)
    if not (0.0 < b < 1.0): return None
    r = y - (a + b * x)
    return dict(b=b, mu=a / (1 - b), sig=r.std(ddof=2) / np.sqrt(1 - b * b), hl=np.log(2) / (-np.log(b)))

@njit
def positions_from_z(z, k, e, stop):
    n = len(z); pos = np.zeros(n); p = 0.0; blocked = 0.0
    for t in range(n):
        zt = z[t]
        if blocked != 0.0 and abs(zt) < k: blocked = 0.0
        if p == 0.0:
            if blocked == 0.0:
                if zt >= k: p = -1.0
                elif zt <= -k: p = 1.0
        else:
            if p < 0 and zt <= e: p = 0.0
            elif p > 0 and zt >= -e: p = 0.0
            elif abs(zt) >= stop:
                blocked = 1.0 if zt > 0 else -1.0
                p = 0.0
        pos[t] = p
    return pos

def ventanas(index, train_days=TRAIN_DAYS):
    day = index.normalize().values; ud = np.unique(day)
    for j in range(MIN_TRAIN, len(ud) - TEST_DAYS + 1, TEST_DAYS):
        tr = (day >= ud[j - train_days]) & (day < ud[j]); te = (day >= ud[j]) & (day < ud[j + TEST_DAYS])
        yield j, tr, te

In [12]:
def _df(s, mu=None):
    sl = s[:-1] - (s[:-1].mean() if mu is None else mu); ds = np.diff(s)
    d = (sl @ ds) / (sl @ sl); e = ds - d * sl
    return -d, -(d / np.sqrt(e.var(ddof=2) / (sl @ sl)))

def puntaje_cestas(Ltr, cestas):
    sc = np.empty(len(cestas)); coefs = []
    for n, idx in enumerate(cestas):
        A = np.column_stack([np.ones(len(Ltr))] + [Ltr[:, m] for m in idx[1:]])
        coef = np.linalg.lstsq(A, Ltr[:, idx[0]], rcond=None)[0]
        sc[n] = _df(Ltr[:, idx[0]] - A @ coef)[1]
        coefs.append(coef)
    return sc, coefs

def elegir_cestas(sc, cestas, n_max=N_MAX, jacc_max=JACC_MAX):
    elegidas = []
    for n in np.argsort(-sc):
        s = set(cestas[n])
        if all(len(s & set(cestas[m])) / len(s | set(cestas[m])) <= jacc_max for m in elegidas):
            elegidas.append(n)
            if len(elegidas) == n_max: break
    return elegidas

def pesos_cesta(idx, coef, n_act):
    w = np.zeros(n_act); w[idx[0]] = 1.0
    for m, c in zip(idx[1:], coef[1:]): w[m] = -c
    return w

def preparar_seleccion(data, n_max=N_MAX, jacc_max=JACC_MAX, train_days=TRAIN_DAYS):
    Lp = np.log(data).values
    cestas = [c for k in (2, 3, 4) for c in combinations(range(data.shape[1]), k)]
    out = []
    for j, tr, te in ventanas(data.index, train_days):
        sc, coefs = puntaje_cestas(Lp[tr], cestas)
        sel = elegir_cestas(sc, cestas, n_max, jacc_max)
        W = np.column_stack([pesos_cesta(cestas[n], coefs[n], data.shape[1]) for n in sel])
        out.append(dict(window=j, tr=tr, te=te, W=W, cestas=[cestas[n] for n in sel], score=sc[sel]))
    return out

In [15]:
seleccion = preparar_seleccion(data_sel)
print(len(seleccion), "ventanas;", sum(s['W'].shape[1] for s in seleccion), "canastas-ventana")
print("Primera ventana de prueba:", data_sel.index[seleccion[0]['te']][0].date(),
      "| última:", data_sel.index[seleccion[-1]['te']][-1].date())

78 ventanas; 1560 canastas-ventana
Primera ventana de prueba: 2010-12-03 | última: 2019-11-22


## Fase 2: Selección de canastas y operación en el bloque de diseño

Aquí se toman **todas las decisiones de diseño**, usando solo el bloque 2010–2019. Se hace en cuatro pasos:

- **2a. Vida media y `HL_MAX`.** Medimos qué tan rápido revierten las canastas elegidas (en días). Con una ventana de prueba de 30 días, una canasta que tarda más de ~10 días en recorrer la mitad del camino de vuelta casi no completa operaciones antes de que termine la ventana (y la liquidación forzada cuesta), así que fijamos *a priori* `HL_MAX ≤ TEST_DAYS / 3 = 10 días`. El diagnóstico dice cuántas canastas quedarían fuera.
- **2b. Operación con centro fijo y centro móvil.** Retraso de una barra (un día), costo de 1 pb por pata, umbrales OU con costo. Se reportan bruto, neto, break-even y p-valor por ventana.
- **2c. Sensibilidad a la configuración.** Umbrales fijos, ventana del centro móvil. Se fija aquí la **única** configuración que se lleva a la prueba final, con una regla escrita antes de ver los resultados.
- **2d. Persistencia de la cointegración** (diagnóstico, como en *Amplitud*), ahora con decenas de ventanas.

*Entrega:* una configuración fijada para la Fase 4.

### 2a. Vida media de las canastas elegidas

La vida media (en días) es el tiempo que tarda el spread en recorrer la mitad del camino de regreso a su centro. Medimos la de cada canasta elegida con sus datos de entrenamiento.

In [16]:
Lp = np.log(data_sel).values
hl = []
for s in seleccion:
    for i in range(s['W'].shape[1]):
        ou = fit_ou(Lp[s['tr']] @ s['W'][:, i])
        hl.append(np.nan if ou is None else ou['hl'])
hl = pd.Series(hl)

print(f"Sin ajuste OU válido (b fuera de (0,1)): {hl.isna().mean():.1%}")
print(hl.describe().round(1))
print()
for h in (5, 10, 15, 20, 30, 45, 60):
    print(f"HL_MAX = {h:>2} días: acepta {(hl <= h).mean():.1%} de las canastas-ventana")

Sin ajuste OU válido (b fuera de (0,1)): 3.6%
count    1504.0
mean        1.0
std         0.6
min         0.1
25%         0.6
50%         0.9
75%         1.4
max         4.4
dtype: float64

HL_MAX =  5 días: acepta 96.4% de las canastas-ventana
HL_MAX = 10 días: acepta 96.4% de las canastas-ventana
HL_MAX = 15 días: acepta 96.4% de las canastas-ventana
HL_MAX = 20 días: acepta 96.4% de las canastas-ventana
HL_MAX = 30 días: acepta 96.4% de las canastas-ventana
HL_MAX = 45 días: acepta 96.4% de las canastas-ventana
HL_MAX = 60 días: acepta 96.4% de las canastas-ventana


**Chequeo contra el azar.** Repetimos la selección y la medición de vida media sobre una caminata aleatoria con la misma covarianza que las divisas (sin cointegración). Si el azar también produce vidas medias de ~1 día, la selección elige spreads "rápidos" por construcción; si produce vidas mucho más largas, las canastas reales revierten más rápido de lo que cabría por selección.

In [17]:
def hl_de(sel, data):
    Lp = np.log(data).values; h = []
    for s in sel:
        for i in range(s['W'].shape[1]):
            ou = fit_ou(Lp[s['tr']] @ s['W'][:, i])
            h.append(np.nan if ou is None else ou['hl'])
    return pd.Series(h)

def simular_rw(data, seed):
    rng = np.random.default_rng(seed); R = np.log(data).diff().dropna().values
    L = np.linalg.cholesky(np.cov(R.T) + 1e-18 * np.eye(R.shape[1]))
    Z = rng.standard_normal(R.shape) @ L.T
    return pd.DataFrame(np.exp(np.log(data.iloc[0].values) + np.vstack([np.zeros(R.shape[1]), np.cumsum(Z, 0)])),
                        index=data.index, columns=data.columns)

HL_MAX = 10                                                      # fijado a priori: TEST_DAYS / 3
sim = simular_rw(data_sel, seed=1)
hl_sim = hl_de(preparar_seleccion(sim), sim)
print(f"Placebo (caminata aleatoria): sin ajuste válido {hl_sim.isna().mean():.1%}")
print(pd.DataFrame({'real': hl.describe(), 'placebo': hl_sim.describe()}).round(1))

Placebo (caminata aleatoria): sin ajuste válido 0.0%
         real  placebo
count  1504.0   1560.0
mean      1.0      2.0
std       0.6      0.5
min       0.1      0.9
25%       0.6      1.7
50%       0.9      2.0
75%       1.4      2.3
max       4.4      5.9


### 2b. Operación con centro fijo y centro móvil

Se opera cada canasta elegida con umbrales OU que descuentan el costo (si ningún umbral gana tras costos, la canasta se abstiene). Se prueban dos centros:
- **Fijo:** el centro es la media del entrenamiento.
- **Móvil:** el centro es la media de las últimas 5 barras (`VENT = 5`, unas 5 vidas medias, como en los datos horarios).

Cada centro se evalúa con **dos retrasos**, fijados a priori como cotas:
- `retraso = 0`: se ejecuta al mismo cierre que genera la señal (cota superior, optimista).
- `retraso = 1`: se ejecuta un día después (cota inferior, muy conservadora: con vida media de ~1 día equivale a esperar una vida media).

La realidad, con ejecución en horas, queda entre ambas. Solo concluimos si el resultado es positivo **incluso con `retraso = 1`**. Reportamos bruto, neto, p-valor por ventana (78 ventanas), operaciones y break-even.

In [18]:
VENT = 5                                           # ventana de la media móvil, en días

def ou_band_search(b, c_side_z, n=40000, seed=0, ks=np.arange(0.75, 3.01, 0.25), es=(0.0, 0.25, 0.5, 1.0)):
    from scipy.signal import lfilter
    eps = np.random.default_rng(seed).standard_normal(n) * np.sqrt(1 - b * b)
    z = lfilter([1.0], [1.0, -b], eps); dz = np.diff(z); best = None
    for k in ks:
        for e in es:
            if e >= k: continue
            pos = positions_from_z(z, k, e, STOP)
            rate = ((pos[:-1] * dz).sum() - c_side_z * np.abs(np.diff(pos, prepend=0.0)).sum()) / n
            if best is None or rate > best[2]: best = (k, e, rate)
    return best if best and best[2] > 0 else None

def run_spread_c(Lp_tr, Lp_te, w, centro='movil', cost_bps=COST_BPS, mode='ou', fixed_ke=(2.0, 0.5), seed=0, retraso=0):
    s_tr, s_te = Lp_tr @ w, Lp_te @ w
    E = np.abs(w).sum()
    if centro == 'fijo':
        ou = fit_ou(s_tr)
        if ou is None or ou['hl'] > HL_MAX: return None
        z = (s_te - ou['mu']) / ou['sig']
    else:
        m_tr = pd.Series(s_tr).rolling(VENT).mean().shift(1).values
        ou = fit_ou((s_tr - m_tr)[VENT:])
        if ou is None or ou['hl'] > HL_MAX: return None
        m = pd.Series(np.concatenate([s_tr[-VENT:], s_te])).rolling(VENT).mean().shift(1).values[VENT:]
        z = (s_te - m) / ou['sig']
    if mode == 'fixed':
        k, e = fixed_ke
    else:
        res = ou_band_search(ou['b'], (cost_bps / 1e4) * E / ou['sig'], seed=seed)
        if res is None: return None
        k, e = res[0], res[1]
    pos = positions_from_z(z, k, e, STOP)
    gross = (pos[:len(pos) - 1 - retraso] * np.diff(s_te)[retraso:]).sum()     # retraso>0: se ejecuta con precios de barras posteriores
    cost = np.abs(np.diff(pos, prepend=0.0)).sum() * E * cost_bps / 1e4
    ops = int(((pos[1:] != 0) & (pos[:-1] == 0)).sum() + (pos[0] != 0))
    return dict(gross=gross / E, cost=cost / E, net=(gross - cost) / E, ops=ops, k=k, e=e, hl=ou['hl'])

def run_ruta_A(data, seleccion, centro='movil', mode='ou', cost_bps=COST_BPS, seed=0, retraso=0,
               fixed_ke=(2.0, 0.5), detalle=False):
    Lp = np.log(data).values; rows, det = [], []
    for s in seleccion:
        res = [run_spread_c(Lp[s['tr']], Lp[s['te']], s['W'][:, i], centro, cost_bps, mode,
                            fixed_ke=fixed_ke, seed=seed, retraso=retraso) for i in range(s['W'].shape[1])]
        ok = [r for r in res if r is not None]; N = s['W'].shape[1]
        det += [dict(r, window=s['window']) for r in ok]
        rows.append(dict(window=s['window'], gross=sum(r['gross'] for r in ok) / N, cost=sum(r['cost'] for r in ok) / N,
                         net=sum(r['net'] for r in ok) / N, ops=sum(r['ops'] for r in ok), n_spreads=len(ok), n_tot=N))
    out = pd.DataFrame(rows)
    return (out, pd.DataFrame(det)) if detalle else out

def summarize(res, label):
    g, n = res.gross, res.net
    be = g.mean() / res.cost.mean() if res.cost.mean() > 0 else np.nan
    return pd.Series({'bruto': g.mean(), 'p_bruto': stats.ttest_1samp(g, 0).pvalue, 'neto': n.mean(),
        'p_neto': stats.ttest_1samp(n, 0).pvalue, 'neto>0': f"{(n>0).sum()}/{len(n)}", 'ops/vent': res.ops.mean(),
        'break-even (x costo)': be}, name=label)

In [19]:
filas = []
for centro in ('fijo', 'movil'):
    for retraso in (0, 1):
        r = run_ruta_A(data_sel, seleccion, centro, 'ou', retraso=retraso)
        filas.append(summarize(r, f'{centro}_ou_retraso{retraso}'))
display(pd.DataFrame(filas)[['bruto', 'p_bruto', 'neto', 'p_neto', 'neto>0', 'ops/vent', 'break-even (x costo)']].round(5))

,bruto,p_bruto,neto,p_neto,neto>0,ops/vent,break-even (x costo)
fijo_ou_retraso0,0.00671,0.00001,0.00625,0.00002,52/78,50.52564,14.59036
fijo_ou_retraso1,-0.00041,0.41874,-0.00087,0.08560,31/78,50.52564,-0.88215
movil_ou_retraso0,0.01014,0.00000,0.00934,0.00000,55/78,85.65385,12.58320
movil_ou_retraso1,-0.00109,0.14452,-0.00190,0.01267,22/78,85.65385,-1.35202


In [20]:
filas = []
for retraso in (0, 1):
    _, det = run_ruta_A(data_sel, seleccion, 'fijo', 'ou', retraso=retraso, detalle=True)
    det['hl_tramo'] = pd.cut(det.hl, [0, 0.5, 1, 2, 10], labels=['<0.5 d', '0.5-1 d', '1-2 d', '2-10 d'])
    t = det.groupby('hl_tramo').agg(n=('gross', 'size'), bruto=('gross', 'mean'), neto=('net', 'mean'))
    t['retraso'] = retraso
    filas.append(t.reset_index())
display(pd.concat(filas).set_index(['retraso', 'hl_tramo']).round(5))

n    bruto     neto
retraso hl_tramo                       
0       <0.5 d    234  0.01345  0.01287
        0.5-1 d   614  0.00935  0.00886
        1-2 d     576  0.00278  0.00234
        2-10 d     80 -0.00026 -0.00064
1       <0.5 d    234 -0.00288 -0.00347
        0.5-1 d   614 -0.00002 -0.00050
        1-2 d     576  0.00010 -0.00034
        2-10 d     80 -0.00006 -0.00043

### 2c. ¿Está el artefacto en las monedas emergentes o es general?

Si la ganancia con `retraso = 0` viene de cotizaciones desfasadas de mercados poco líquidos, debería concentrarse en las canastas con monedas emergentes y ser menor en las que solo tienen monedas del G10. Clasificamos cada canasta según cuántas monedas emergentes contiene y comparamos el bruto con ambos retrasos.

In [22]:
G10 = ['Euro', 'Libra', 'Yen', 'Franco_SUI', 'Dolar_CAN', 'Dolar_AUS', 'Dolar_NZ', 'Corona_NO', 'Corona_SE']
Lp = np.log(data_sel).values
filas = []
for s in seleccion:
    for i in range(s['W'].shape[1]):
        n_em = sum(data_sel.columns[j] not in G10 for j in s['cestas'][i])
        for retraso in (0, 1):
            r = run_spread_c(Lp[s['tr']], Lp[s['te']], s['W'][:, i], 'fijo', mode='ou', retraso=retraso)
            if r is not None:
                filas.append(dict(retraso=retraso,
                                  tipo=('solo G10' if n_em == 0 else '1 emergente' if n_em == 1 else '2+ emergentes'),
                                  bruto=r['gross'], neto=r['net']))
D = pd.DataFrame(filas)
display(D.groupby(['retraso', 'tipo']).agg(n=('bruto', 'size'), bruto=('bruto', 'mean'), neto=('neto', 'mean')).round(5))

n    bruto     neto
retraso tipo                                 
0       1 emergente     264  0.00521  0.00475
        2+ emergentes  1220  0.00744  0.00696
        solo G10         20  0.00068  0.00025
1       1 emergente     264 -0.00047 -0.00093
        2+ emergentes  1220 -0.00043 -0.00091
        solo G10         20  0.00060  0.00017

### 2d. Universo restringido al G10

El diagnóstico anterior mostró que la ganancia con `retraso = 0` se concentra en las canastas con monedas emergentes (cotizaciones desfasadas) y que las canastas solo-G10 conservan su bruto con un día de retraso. Repetimos el procedimiento completo con **solo las 9 monedas del G10** (Euro, Libra, Yen, Franco suizo, dólares canadiense, australiano y neozelandés, coronas noruega y sueca).

Con menos canastas posibles (~250 contra ~4,000) la selección se parece menos a "la mejor entre miles". La regla se mantiene: solo concluimos si el neto es positivo **con `retraso = 1`**.

In [23]:
G10_ok = [c for c in G10 if c in data_sel.columns]
print("Monedas G10 disponibles:", len(G10_ok), G10_ok)
data_g10 = data_sel[G10_ok]

sel_g10 = preparar_seleccion(data_g10)
print(len(sel_g10), "ventanas;", round(np.mean([s['W'].shape[1] for s in sel_g10]), 1), "canastas por ventana (media)")

filas = []
for centro in ('fijo', 'movil'):
    for retraso in (0, 1):
        r = run_ruta_A(data_g10, sel_g10, centro, 'ou', retraso=retraso)
        filas.append(summarize(r, f'G10_{centro}_retraso{retraso}'))
display(pd.DataFrame(filas)[['bruto', 'p_bruto', 'neto', 'p_neto', 'neto>0', 'ops/vent', 'break-even (x costo)']].round(5))

Monedas G10 disponibles: 9 ['Euro', 'Libra', 'Yen', 'Franco_SUI', 'Dolar_CAN', 'Dolar_AUS', 'Dolar_NZ', 'Corona_NO', 'Corona_SE']
78 ventanas; 19.9 canastas por ventana (media)


,bruto,p_bruto,neto,p_neto,neto>0,ops/vent,break-even (x costo)
G10_fijo_retraso0,0.00003,0.92609,-0.00027,0.42385,36/78,34.34615,0.10418
G10_fijo_retraso1,0.00000,0.99776,-0.00030,0.38239,38/78,34.34615,0.00321
G10_movil_retraso0,0.00048,0.29315,-0.00029,0.52589,39/78,82.07692,0.62695
G10_movil_retraso1,0.00033,0.53049,-0.00044,0.40198,35/78,82.07692,0.42980


## Conclusión de Datos Diarios

**Qué se hizo.** Se repitió el procedimiento de *Amplitud* con datos diarios desde 2010 y 18 divisas, separando un bloque de diseño (2010–2019, 78 ventanas) de un bloque de prueba (2020 en adelante). Las decisiones se tomaron solo con el primero, con la regla de concluir únicamente si el neto era positivo con un día de retraso.

**Qué se encontró.**
1. Las canastas elegidas revierten más rápido que el azar (vida media mediana de 0.9 días contra 2.0 del placebo).
2. Con ejecución en la misma barra el resultado es muy alto (break-even de 12–15 veces el costo), pero con un día de retraso el bruto desaparece (fijo: −0.0004, p = 0.42) y en el centro móvil el neto es significativamente negativo (p = 0.013).
3. El bruto con `retraso = 0` es mayor cuanto más rápida es la reversión y desaparece con el retraso en todos los tramos: no es la huella de una reversión económica sino de ruido de observación (cierres diarios a distintas horas y liquidez).
4. El efecto crece con el número de monedas emergentes de la canasta (0.0007 en solo G10, 0.0052 con 1 emergente, 0.0074 con 2+). Los puntajes de estacionariedad con los que se seleccionan canastas lo premian, y por eso 99% de los spreads elegidos contenían emergentes.
5. Restringido al G10, el bruto es ≈ 0 (p > 0.29) y el neto es negativo en las cuatro configuraciones.

**Qué significa.** Con datos públicos de yfinance a frecuencia horaria o diaria no se encontró una ganancia neta operable. La ganancia aparente proviene de la sincronización de cotizaciones y desaparece con un retraso realista.

**Decisión de diseño.** Ninguna configuración superó la regla en el bloque de diseño, así que el bloque de prueba (2020–2026) no se utilizó. Queda disponible, sin contaminar, para evaluar cualquier propuesta futura.

**Limitaciones.** Costo de 1 pb por pata para todas las monedas; datos diarios de cierre no sincronizados; un solo universo y un solo conjunto de ventanas; solo se evaluaron las estrategias de reversión descritas.